# Q1: Delivery Performance Prediction
## Can we predict exactly how long a delivery will take — before we place the order?

**Report Section:** Question 1 · Regression  
**Data Source:** Supply Chain ML Predictive Modelling Report

## Objective

- Build a regression model to forecast `delivery_time_days` using pre-order and supplier features
- Identify which factors most strongly influence delivery time and quantify prediction accuracy
- Translate statistical performance into operational business error thresholds (±2 days acceptable, ±5 days review, >5 days critical)

In [ ]:
# Step 0: Setup & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load dataset
# df = pd.read_csv('supply_chain_data.csv')  # Replace with actual data path
# print(df.head())
# print(df.info())
# print(df['delivery_time_days'].describe())

## Implementation Plan

### Step 1: Define Target & Understand Distribution
- Set `delivery_time_days` as regression target (y)
- Plot histogram with KDE overlay to check for skewness and outliers
- Report mean, median, std, IQR; consider log-transform if heavily right-skewed

### Step 2: Feature Selection & Preprocessing Pipeline
- Predictors: `on_time_delivery_rate`, `lead_time_variance`, `delivery_mode`, `delivery_term_code`, `order_frequency_monthly`, `avg_order_volume`, `items_requested`, `items_offered`, `temporal_month`
- OneHotEncode categoricals, StandardScale numerics
- Wrap in ColumnTransformer inside Pipeline to prevent data leakage

### Step 3: Train-Test Split & Baseline Model
- 80/20 split with random_state=42
- Train LinearRegression baseline; compute MAE, RMSE, R²
- Visualize Predicted vs. Actual with 45° reference line

### Step 4: Upgrade to Random Forest Regressor
- Replace with RandomForestRegressor(n_estimators=100, random_state=42)
- Compare metrics using 5-fold cross_val_score
- Plot MAE & R² comparison

### Step 5: Feature Importance Analysis
- Extract `feature_importances_` from fitted RandomForest
- Plot horizontal bar chart (top 10)
- Highlight operational levers (e.g., if lead_time_variance dominates, focus on supplier standardisation)

### Step 6: Residual Analysis & Business Error Thresholds
- Compute residuals = Actual - Predicted
- Define thresholds: ±2 days = acceptable, ±5 days = review, >5 days = critical
- Report % predictions in each band as boardroom-ready summary

In [ ]:
# STEP 1: Define Target & Understand Distribution

# y = df['delivery_time_days']
# fig, ax = plt.subplots(1, 1, figsize=(10, 5))
# y.hist(bins=30, kde=True, ax=ax, alpha=0.7, edgecolor='black')
# ax.set_xlabel('Delivery Time (Days)')
# ax.set_ylabel('Frequency')
# ax.set_title('Distribution of delivery_time_days')
# plt.tight_layout()
# plt.show()

# print(f"Mean: {y.mean():.2f}")
# print(f"Median: {y.median():.2f}")
# print(f"Std Dev: {y.std():.2f}")
# print(f"IQR: {y.quantile(0.75) - y.quantile(0.25):.2f}")
# print(f"Skewness: {y.skew():.3f}")

In [ ]:
# STEP 2 & 3: Feature Selection, Preprocessing, Train-Test Split & Baseline

# Define feature groups
# numerical_features = ['on_time_delivery_rate', 'lead_time_variance', 'order_frequency_monthly', 
#                       'avg_order_volume', 'items_requested', 'items_offered']
# categorical_features = ['delivery_mode', 'delivery_term_code', 'temporal_month']

# X = df[numerical_features + categorical_features]
# y = df['delivery_time_days']

# # Create preprocessing pipeline
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', StandardScaler(), numerical_features),
#         ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
#     ])

# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Linear Regression Baseline
# baseline_pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', LinearRegression())
# ])

# baseline_pipeline.fit(X_train, y_train)
# y_pred_baseline = baseline_pipeline.predict(X_test)

# # Metrics
# mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
# rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
# r2_baseline = r2_score(y_test, y_pred_baseline)

# print(f"Linear Regression Baseline:")
# print(f"  MAE:  {mae_baseline:.3f}")
# print(f"  RMSE: {rmse_baseline:.3f}")
# print(f"  R²:   {r2_baseline:.3f}")

In [ ]:
# STEP 3 Visualization: Predicted vs. Actual (Baseline)

# fig, ax = plt.subplots(1, 1, figsize=(10, 8))
# ax.scatter(y_test, y_pred_baseline, alpha=0.6, s=50)
# min_val = min(y_test.min(), y_pred_baseline.min())
# max_val = max(y_test.max(), y_pred_baseline.max())
# ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
# ax.set_xlabel('Actual Delivery Time (Days)')
# ax.set_ylabel('Predicted Delivery Time (Days)')
# ax.set_title(f'Linear Regression: Predicted vs. Actual (R² = {r2_baseline:.3f})')
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 4: Upgrade to Random Forest Regressor

# rf_pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
# ])

# rf_pipeline.fit(X_train, y_train)
# y_pred_rf = rf_pipeline.predict(X_test)

# # Metrics
# mae_rf = mean_absolute_error(y_test, y_pred_rf)
# rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
# r2_rf = r2_score(y_test, y_pred_rf)

# # Cross-validation
# cv_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='r2')

# print(f"Random Forest Results:")
# print(f"  MAE:  {mae_rf:.3f}")
# print(f"  RMSE: {rmse_rf:.3f}")
# print(f"  R²:   {r2_rf:.3f}")
# print(f"\nCross-Validation R² (5-fold):")
# print(f"  Mean: {cv_scores.mean():.3f}")
# print(f"  Std:  {cv_scores.std():.3f}")

In [ ]:
# STEP 4 Visualization: Model Comparison

# comparison_df = pd.DataFrame({
#     'Metric': ['MAE', 'RMSE', 'R²'],
#     'Linear Regression': [mae_baseline, rmse_baseline, r2_baseline],
#     'Random Forest': [mae_rf, rmse_rf, r2_rf]
# })

# fig, ax = plt.subplots(1, 1, figsize=(10, 6))
# x = np.arange(len(comparison_df))
# width = 0.35
# ax.bar(x - width/2, comparison_df['Linear Regression'], width, label='Linear Regression')
# ax.bar(x + width/2, comparison_df['Random Forest'], width, label='Random Forest')
# ax.set_xlabel('Metric')
# ax.set_ylabel('Score')
# ax.set_title('Model Performance Comparison')
# ax.set_xticks(x)
# ax.set_xticklabels(comparison_df['Metric'])
# ax.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 5: Feature Importance Analysis

# # Get feature names after preprocessing
# feature_names = (numerical_features + 
#                  [f"{cat}_{cat_val}" for cat in categorical_features 
#                   for cat_val in df[cat].unique()[1:]])

# # Extract importance
# rf_model = rf_pipeline.named_steps['model']
# importance_df = pd.DataFrame({
#     'Feature': feature_names[:len(rf_model.feature_importances_)],
#     'Importance': rf_model.feature_importances_
# }).sort_values('Importance', ascending=True).tail(10)

# fig, ax = plt.subplots(1, 1, figsize=(10, 6))
# ax.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
# ax.set_xlabel('Importance Score')
# ax.set_title('Top 10 Features Driving Delivery Time')
# plt.tight_layout()
# plt.show()

# print(importance_df)

In [ ]:
# STEP 6: Residual Analysis & Business Error Thresholds

# residuals = y_test - y_pred_rf

# # Define thresholds
# threshold_acceptable = 2
# threshold_review = 5

# acceptable = np.sum(np.abs(residuals) <= threshold_acceptable) / len(residuals) * 100
# review = np.sum((np.abs(residuals) > threshold_acceptable) & (np.abs(residuals) <= threshold_review)) / len(residuals) * 100
# critical = np.sum(np.abs(residuals) > threshold_review) / len(residuals) * 100

# threshold_summary = pd.DataFrame({
#     'Error Band': ['±2 days (Acceptable)', '±2 to ±5 days (Review)', '>5 days (Critical)'],
#     'Percentage': [f"{acceptable:.1f}%", f"{review:.1f}%", f"{critical:.1f}%"],
#     'Count': [
#         np.sum(np.abs(residuals) <= threshold_acceptable),
#         np.sum((np.abs(residuals) > threshold_acceptable) & (np.abs(residuals) <= threshold_review)),
#         np.sum(np.abs(residuals) > threshold_review)
#     ]
# })

# print("\n=== BUSINESS ERROR THRESHOLD SCORECARD ===")
# print(threshold_summary.to_string(index=False))

In [ ]:
# STEP 6 Visualization: Residual Analysis

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Residual histogram
# axes[0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
# axes[0].axvline(x=-threshold_acceptable, color='orange', linestyle='--', label=f'±{threshold_acceptable} day bands')
# axes[0].axvline(x=threshold_acceptable, color='orange', linestyle='--')
# axes[0].axvline(x=-threshold_review, color='red', linestyle='--', label=f'±{threshold_review} day bands')
# axes[0].axvline(x=threshold_review, color='red', linestyle='--')
# axes[0].set_xlabel('Residual (Days)')
# axes[0].set_ylabel('Frequency')
# axes[0].set_title('Residual Distribution')
# axes[0].legend()

# # Residual vs Predicted
# axes[1].scatter(y_pred_rf, residuals, alpha=0.6, s=50)
# axes[1].axhline(y=0, color='k', linestyle='-', lw=1)
# axes[1].axhline(y=threshold_acceptable, color='orange', linestyle='--')
# axes[1].axhline(y=-threshold_acceptable, color='orange', linestyle='--')
# axes[1].axhline(y=threshold_review, color='red', linestyle='--')
# axes[1].axhline(y=-threshold_review, color='red', linestyle='--')
# axes[1].set_xlabel('Predicted Delivery Time (Days)')
# axes[1].set_ylabel('Residual (Days)')
# axes[1].set_title('Residuals vs. Predicted Values')
# axes[1].grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

## Key Takeaways

Once all steps are complete, your notebook will deliver:

1. **Target Distribution Insight** — Mean, median, and distribution shape of delivery times
2. **Baseline Performance** — Linear regression MAE, RMSE, R² as the interpretable lower bound
3. **Model Upgrade** — Random forest lift over baseline, with cross-validation confirmation
4. **Operational Levers** — Top 5 features driving delivery time (e.g., lead_time_variance, delivery_mode)
5. **Business SLA Alignment** — % predictions within ±2 days (acceptable), ±2–5 days (review), >5 days (critical)

This translates statistical models into procurement and logistics decisions.